In [1]:
# --- utils_load_and_split.py (put in a cell) -------------------------------
import os, re, numpy as np, pandas as pd

def _basename_no_ext(p):
    b = os.path.basename(str(p))
    return re.sub(r"\.[A-Za-z0-9]+$", "", b)

def _load_npz_as_table(npz_path, source_tag):
    """
    Loads an NPZ where each KEY is an image path and each VALUE is a 1D embedding vector.
    Returns:
      df_npz: columns [source, img_key, emb_index, image_id, X_block, X_ptr]
      X: np.ndarray of shape [N, D]
    """
    npz = np.load(npz_path, allow_pickle=True)
    ids = list(npz.keys())
    X = np.vstack([npz[k] for k in ids])
    df_npz = pd.DataFrame({
        "source": source_tag,
        "img_key": [_basename_no_ext(k).lower() for k in ids],
        "emb_index": np.arange(len(ids), dtype=int),
        "image_id": [_basename_no_ext(k) for k in ids],
        "X_block": source_tag,
        "X_ptr": np.arange(len(ids), dtype=int)
    })
    return df_npz, X

def _attach_labels(df_npz, labels_csv, id_col_name, LABELS):
    """
    Merge NPZ table with label CSV on normalized image id.
    - id_col_name: the column in the CSV that holds image id (e.g., 'img_id' for PAD, 'image_id' for derm12345).
    Returns merged dataframe filtered to rows that have ANY of the LABELS.
    """
    lab = pd.read_csv(labels_csv)
    if id_col_name not in lab.columns:
        raise ValueError(f"'{id_col_name}' not in {labels_csv}. Available: {list(lab.columns)}")
    lab["img_key"] = lab[id_col_name].astype(str).map(lambda s: _basename_no_ext(s).lower())

    need_cols = ["patient_id","img_key"] + LABELS
    miss = [c for c in need_cols if c not in lab.columns]
    if miss:
        raise ValueError(f"Missing columns in {labels_csv}: {miss}")

    merged = df_npz.merge(lab[need_cols], on="img_key", how="left")
    keep_mask = merged[LABELS].notna().any(axis=1)
    merged = merged[keep_mask].reset_index(drop=True)
    # force ints for labels
    merged[LABELS] = merged[LABELS].fillna(0).astype(int)
    return merged

def _pull_X_from_blocks(df_all, X_blocks):
    """Row-wise gather embeddings from the correct block/index."""
    return np.vstack([X_blocks[row["X_block"]][row["X_ptr"]] for _, row in df_all.iterrows()])

def load_datasets(
    use_pad=True,
    use_derm12345=True,
    PAD_NPZ_PATH="dermatology_pad_precomputed_embeddings.npz",
    PAD_LABELS_CSV="PAD_labels.csv",
    DERM_NPZ_PATH="derm12345_google-derm-foundation-model_embeddings.npz",
    DERM_LABELS_CSV="derm12345_labels.csv",
    LABELS=("NEV","BCC","ACK","SEK","SCC","MEL"),
):
    """
    Loads selected datasets, maps embeddings to labels, and returns:
      X            : ndarray [N, D]
      y_onehot     : ndarray [N, C] in the order of LABELS
      df_all       : dataframe with columns (patient_id, image_id, source, LABELS, etc.)
      LABELS       : tuple of label names (for reference/order)
    """
    parts = []
    X_blocks = {}

    if use_pad:
        df_pad, X_pad = _load_npz_as_table(PAD_NPZ_PATH, source_tag="PAD")
        df_pad = _attach_labels(df_pad, PAD_LABELS_CSV, id_col_name="img_id", LABELS=list(LABELS))
        parts.append(df_pad)
        X_blocks["PAD"] = X_pad

    if use_derm12345:
        df_derm, X_derm = _load_npz_as_table(DERM_NPZ_PATH, source_tag="DERM")
        df_derm = _attach_labels(df_derm, DERM_LABELS_CSV, id_col_name="image_id", LABELS=list(LABELS))
        parts.append(df_derm)
        X_blocks["DERM"] = X_derm

    if not parts:
        raise ValueError("No dataset selected. Set use_pad and/or use_derm12345 to True.")

    df_all = pd.concat(parts, ignore_index=True)
    X = _pull_X_from_blocks(df_all, X_blocks)
    y_onehot = df_all[list(LABELS)].astype(int).values
    return X, y_onehot, df_all, tuple(LABELS)

def patient_level_split(df_all, LABELS=("NEV","BCC","ACK","SEK","SCC","MEL"), random_state=42):
    """
    Creates a 70/15/15 split at the PATIENT level using multilabel stratification if available;
    falls back to random patient split if iterstrat isn't installed.
    Returns:
      masks: dict with boolean masks { 'train', 'val', 'test' } for df_all rows
      per_patient: dataframe with patient-level multi-hot and assigned split
    """
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
        has_ml_strat = True
    except Exception:
        has_ml_strat = False

    # Build per-patient multi-hot
    per_patient = (
        df_all[["patient_id"] + list(LABELS)]
        .groupby("patient_id")[list(LABELS)].max()
        .reset_index()
    )
    patients = per_patient["patient_id"].astype(str).values
    Yp = per_patient[list(LABELS)].astype(int).values

    rng = np.random.RandomState(random_state)
    if has_ml_strat:
        mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
        tr_pat_idx, temp_pat_idx = next(mskf.split(np.zeros((len(patients),1)), Yp))
        # split temp into val/test
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold as MSKF2
        Ytemp = Yp[temp_pat_idx]
        mskf2 = MSKF2(n_splits=2, shuffle=True, random_state=random_state+1)
        val_rel, test_rel = next(mskf2.split(np.zeros((len(temp_pat_idx),1)), Ytemp))
        val_pat_idx = temp_pat_idx[val_rel]
        test_pat_idx = temp_pat_idx[test_rel]
    else:
        idx = np.arange(len(patients))
        rng.shuffle(idx)
        n = len(idx)
        tr_pat_idx = idx[:int(0.7*n)]
        val_pat_idx = idx[int(0.7*n):int(0.85*n)]
        test_pat_idx= idx[int(0.85*n):]

    # Map patient split back to row masks
    pat_to_split = {}
    for i in tr_pat_idx:   pat_to_split[patients[i]] = "train"
    for i in val_pat_idx:  pat_to_split[patients[i]] = "val"
    for i in test_pat_idx: pat_to_split[patients[i]] = "test"

    df_all = df_all.copy()
    df_all["split"] = df_all["patient_id"].astype(str).map(pat_to_split)

    masks = {
        "train": (df_all["split"] == "train").values,
        "val":   (df_all["split"] == "val").values,
        "test":  (df_all["split"] == "test").values,
    }

    # also return annotated per_patient table
    per_patient = per_patient.copy()
    per_patient["split"] = per_patient["patient_id"].astype(str).map(pat_to_split)
    return masks, per_patient


In [2]:
# --- example_load_call.py (put in the next cell) ---------------------------
# Config flags
USE_PAD = True
USE_DERM12345 = True

LABELS = ("BCC","ACK","SEK","SCC","MEL")

# Load data
X, y_onehot, df_all, LABELS = load_datasets(
    use_pad=USE_PAD,
    use_derm12345=USE_DERM12345,
    PAD_NPZ_PATH="dermatology_pad_precomputed_embeddings.npz",
    PAD_LABELS_CSV="PAD_labels.csv",
    DERM_NPZ_PATH="derm12345_google-derm-foundation-model_embeddings.npz",
    DERM_LABELS_CSV="derm12345_labels.csv",
    LABELS=LABELS
)

# Patient-level 70/15/15 split
masks, per_patient = patient_level_split(df_all, LABELS=LABELS, random_state=42)

print("Shapes: X =", X.shape, "| y_onehot =", y_onehot.shape)
print("Rows per split:", {k:int(v.sum()) for k,v in masks.items()})
print("Patients per split:", per_patient.groupby("split")["patient_id"].nunique().to_dict())

# (Optional) Indices for convenience
import numpy as np
train_idx = np.where(masks["train"])[0]
val_idx   = np.where(masks["val"])[0]
test_idx  = np.where(masks["test"])[0]

# (Optional) quick class counts by split
for split, mask in masks.items():
    counts = y_onehot[mask].sum(axis=0)
    print(f"{split} class counts:", dict(zip(LABELS, map(int, counts))))


Shapes: X = (3808, 6144) | y_onehot = (3808, 5)
Rows per split: {'train': 2996, 'val': 391, 'test': 421}
Patients per split: {'test': 172, 'train': 1350, 'val': 164}
train class counts: {'BCC': 1029, 'ACK': 625, 'SEK': 651, 'SCC': 358, 'MEL': 333}
val class counts: {'BCC': 116, 'ACK': 78, 'SEK': 83, 'SCC': 55, 'MEL': 59}
test class counts: {'BCC': 123, 'ACK': 85, 'SEK': 108, 'SCC': 45, 'MEL': 60}


In [3]:
# --- multinomial_logreg_eval.py --------------------------------------------
import time, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

CLASS_NAMES = list(LABELS)  # uses your current LABELS from the previous cell

# y indices from one-hot
y_idx = y_onehot.argmax(axis=1)

# split views
train_idx = np.where(masks["train"])[0]
val_idx   = np.where(masks["val"])[0]
test_idx  = np.where(masks["test"])[0]

X_train, y_train = X[train_idx], y_idx[train_idx]
X_val,   y_val   = X[val_idx],   y_idx[val_idx]
X_test,  y_test  = X[test_idx],  y_idx[test_idx]

# model
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    ))
])

t0 = time.time()
print("[LogReg] training...")
model.fit(X_train, y_train)
print(f"[LogReg] trained in {time.time()-t0:.1f}s")

# probabilities on test
proba_test = model.predict_proba(X_test)
y_pred = proba_test.argmax(axis=1)

# classification report
print("\n=== Classification report (TEST) ===")
print(classification_report(
    y_test, y_pred,
    labels=np.arange(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    digits=3,
    zero_division=0
))

# per-class AUC (one-vs-rest) + macro AUC
print("=== ROC-AUC (TEST) ===")
per_class = {}
vals = []
for i, cname in enumerate(CLASS_NAMES):
    y_bin = (y_test == i).astype(int)
    # skip if class absent in test (AUC undefined)
    if y_bin.min() == y_bin.max():
        per_class[cname] = float("nan")
        print(f"{cname}: nan (class absent in test)")
        continue
    auc_i = roc_auc_score(y_bin, proba_test[:, i])
    per_class[cname] = auc_i
    vals.append(auc_i)
    print(f"{cname}: {auc_i:.3f}")

macro_auc = float(np.mean(vals)) if len(vals) else float("nan")
print(f"Macro AUC: {macro_auc:.3f}" if np.isfinite(macro_auc) else "Macro AUC: nan")

# (optional) scikit's multiclass AUC (macro-OVR) for cross-check
try:
    y_test_binarized = np.eye(len(CLASS_NAMES), dtype=int)[y_test]
    sk_macro_auc = roc_auc_score(y_test_binarized, proba_test, average="macro", multi_class="ovr")
    print(f"Macro AUC (sklearn multiclass ovr): {sk_macro_auc:.3f}")
except Exception as e:
    print(f"sklearn multiclass AUC not available: {e}")


[LogReg] training...


c:\code\DermFoundation\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[LogReg] trained in 4.1s

=== Classification report (TEST) ===
              precision    recall  f1-score   support

         BCC      0.674     0.707     0.690       123
         ACK      0.689     0.729     0.709        85
         SEK      0.890     0.898     0.894       108
         SCC      0.353     0.400     0.375        45
         MEL      0.810     0.567     0.667        60

    accuracy                          0.708       421
   macro avg      0.683     0.660     0.667       421
weighted avg      0.718     0.708     0.709       421

=== ROC-AUC (TEST) ===
BCC: 0.871
ACK: 0.931
SEK: 0.982
SCC: 0.776
MEL: 0.874
Macro AUC: 0.887
Macro AUC (sklearn multiclass ovr): 0.887


In [3]:
# --- train_and_auc_with_reports.py ------------------------------------------
import numpy as np, time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print("⚠️ xgboost not installed; skipping XGBoost.")

# 1) Build y indices and split views
CLASS_NAMES = list(LABELS)  # e.g., ["BCC","ACK","SEK","SCC","MEL"]
y_idx = y_onehot.argmax(axis=1)

train_idx = np.where(masks["train"])[0]
val_idx   = np.where(masks["val"])[0]
test_idx  = np.where(masks["test"])[0]

X_train, y_train = X[train_idx], y_idx[train_idx]
X_val,   y_val   = X[val_idx],   y_idx[val_idx]
X_test,  y_test  = X[test_idx],  y_idx[test_idx]

# 2) Define models (same as yours)
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                   multi_class="multinomial", random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance"))
    ]),
    "SVC (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(probability=True, kernel="rbf", C=1.0, gamma="scale", random_state=42))
    ]),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=400, class_weight="balanced", random_state=42),
}
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", random_state=42)

def per_class_auc(y_true_idx, proba, class_names):
    scores, vals = {}, []
    for i, cname in enumerate(class_names):
        y_bin = (y_true_idx == i).astype(int)
        if y_bin.min() == y_bin.max():  # class absent in test
            scores[cname] = float("nan")
            continue
        auc = roc_auc_score(y_bin, proba[:, i])
        scores[cname] = auc
        vals.append(auc)
    macro = float(np.mean(vals)) if len(vals) else float("nan")
    return scores, macro

def get_proba(model, Xte):
    """Use predict_proba if available, else decision_function -> softmax."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(Xte)
    if isinstance(model, Pipeline):
        clf = model[-1]
        if hasattr(clf, "predict_proba"):
            return clf.predict_proba(model[:-1].transform(Xte))
        if hasattr(clf, "decision_function"):
            D = clf.decision_function(model[:-1].transform(Xte))
        else:
            raise RuntimeError("Model has neither predict_proba nor decision_function.")
    else:
        if hasattr(model, "decision_function"):
            D = model.decision_function(Xte)
        else:
            raise RuntimeError("Model has neither predict_proba nor decision_function.")
    D = np.atleast_2d(D)
    e = np.exp(D - D.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

# 3) Train & evaluate (print report AND AUCs after each model)
all_results = {}
for name, model in models.items():
    t0 = time.time()
    print(f"\n[{name}] training...")
    model.fit(X_train, y_train)
    print(f"[{name}] trained in {time.time()-t0:.2f}s; evaluating...")

    proba = get_proba(model, X_test)
    y_pred = proba.argmax(axis=1)

    # Classification report
    print(f"\n=== Classification report (TEST) — {name} ===")
    print(classification_report(
        y_test, y_pred,
        labels=np.arange(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0
    ))

    # Per-class AUCs + macro
    scores, macro = per_class_auc(y_test, proba, CLASS_NAMES)
    print(f"=== ROC-AUC (TEST) — {name} ===")
    for cname in CLASS_NAMES:
        v = scores.get(cname, float("nan"))
        print(f"{cname}: {v:.3f}" if np.isfinite(v) else f"{cname}: nan")
    print(f"Macro AUC: {macro:.3f}" if np.isfinite(macro) else "Macro AUC: nan")

    # sklearn multiclass macro (OVR) cross-check
    try:
        y_test_bin = np.eye(len(CLASS_NAMES), dtype=int)[y_test]
        sk_macro_auc = roc_auc_score(y_test_bin, proba, average="macro", multi_class="ovr")
        print(f"Macro AUC (sklearn multiclass ovr): {sk_macro_auc:.3f}")
    except Exception as e:
        print(f"Macro AUC (sklearn multiclass ovr): n/a ({e})")

    all_results[name] = (scores, macro)

# 4) Summary AUC table
print("\n=== Per-class ROC-AUC (TEST) — Summary ===")
for name, (scores, macro) in all_results.items():
    parts = []
    for cname in CLASS_NAMES:
        v = scores.get(cname, float("nan"))
        parts.append(f"{cname}:{v:.3f}" if np.isfinite(v) else f"{cname}:nan")
    macro_txt = f"{macro:.3f}" if np.isfinite(macro) else "nan"
    print(f"{name:>20} | " + "  ".join(parts) + f"  ||  Macro:{macro_txt}")

print("\nImages per split ->",
      "Train:", len(train_idx), "| Val:", len(val_idx), "| Test:", len(test_idx))



[Logistic Regression] training...


c:\code\DermFoundation\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[Logistic Regression] trained in 3.13s; evaluating...

=== Classification report (TEST) — Logistic Regression ===
              precision    recall  f1-score   support

         BCC      0.674     0.707     0.690       123
         ACK      0.689     0.729     0.709        85
         SEK      0.890     0.898     0.894       108
         SCC      0.353     0.400     0.375        45
         MEL      0.810     0.567     0.667        60

    accuracy                          0.708       421
   macro avg      0.683     0.660     0.667       421
weighted avg      0.718     0.708     0.709       421

=== ROC-AUC (TEST) — Logistic Regression ===
BCC: 0.871
ACK: 0.931
SEK: 0.982
SCC: 0.776
MEL: 0.874
Macro AUC: 0.887
Macro AUC (sklearn multiclass ovr): 0.887

[Random Forest] training...
[Random Forest] trained in 136.06s; evaluating...

=== Classification report (TEST) — Random Forest ===
              precision    recall  f1-score   support

         BCC      0.599     0.862     0.707       

In [4]:
# --- Run ONLY Logistic Regression from your existing `models` dict, with AUC + 95% CI ---
# relies on variables/functions already defined above: models, X_train, y_train, X_test, y_test,
# CLASS_NAMES, df_all, test_idx, per_class_auc, get_proba

import numpy as np, time
from sklearn.metrics import classification_report

# settings for CI
N_BOOT = 300        # bump to 1000 for tighter CIs (slower)
BY_PATIENT = True   # bootstrap at patient level (recommended)
RNG_SEED = 42
rng = np.random.RandomState(RNG_SEED)

# 1) Pull LR from your existing models dict and train
logreg = models["Logistic Regression"]

t0 = time.time()
logreg.fit(X_train, y_train)
print(f"[LogReg] trained in {time.time()-t0:.2f}s")

# 2) Predict probabilities on TEST
proba_test = get_proba(logreg, X_test)
y_pred = proba_test.argmax(axis=1)

# 3) Classification report
print("\n=== Classification report (TEST) — Logistic Regression ===")
print(classification_report(
    y_test, y_pred,
    labels=np.arange(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    digits=3,
    zero_division=0
))

# 4) Point-estimate AUCs (one-vs-rest) + macro
per_cls_auc, macro_auc = per_class_auc(y_test, proba_test, CLASS_NAMES)

# 5) Bootstrap 95% CIs (patient-aware)
if BY_PATIENT:
    test_pat = df_all.iloc[test_idx]["patient_id"].astype(str).values
    uniq_pats = np.unique(test_pat)
    pat_to_idx = {p: np.where(test_pat == p)[0] for p in uniq_pats}

def bootstrap_auc_ci(y_true, proba, class_names, n_boot=N_BOOT):
    per_cls_samples = {c: [] for c in class_names}
    macro_samples = []
    n = len(y_true)
    for _ in range(n_boot):
        if BY_PATIENT:
            sampled_pats = rng.choice(uniq_pats, size=len(uniq_pats), replace=True)
            idxs = np.concatenate([pat_to_idx[p] for p in sampled_pats])
        else:
            idxs = rng.choice(np.arange(n), size=n, replace=True)
        s_b, m_b = per_class_auc(y_true[idxs], proba[idxs], class_names)
        for c, v in s_b.items():
            if np.isfinite(v):
                per_cls_samples[c].append(v)
        if np.isfinite(m_b):
            macro_samples.append(m_b)

    pct = lambda arr: (np.percentile(arr, 2.5), np.percentile(arr, 97.5)) if len(arr) else (np.nan, np.nan)
    per_cls_ci = {c: pct(per_cls_samples[c]) for c in class_names}
    macro_ci = pct(macro_samples)
    return per_cls_ci, macro_ci

per_cls_ci, macro_ci = bootstrap_auc_ci(y_test, proba_test, CLASS_NAMES, n_boot=N_BOOT)

# 6) Print AUCs with 95% CI
print("=== ROC-AUC (TEST) — Logistic Regression ===")
for cname in CLASS_NAMES:
    auc = per_cls_auc.get(cname, np.nan)
    lo, hi = per_cls_ci.get(cname, (np.nan, np.nan))
    if np.isfinite(auc):
        print(f"{cname}: {auc:.3f}  (95% CI: {lo:.3f}–{hi:.3f})")
    else:
        print(f"{cname}: nan  (95% CI: nan–nan)")

mlo, mhi = macro_ci
macro_txt = f"{macro_auc:.3f}" if np.isfinite(macro_auc) else "nan"
mlo_txt   = f"{mlo:.3f}" if np.isfinite(mlo) else "nan"
mhi_txt   = f"{mhi:.3f}" if np.isfinite(mhi) else "nan"
print(f"Macro AUC: {macro_txt}  (95% CI: {mlo_txt}–{mhi_txt})")


c:\code\DermFoundation\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[LogReg] trained in 3.61s

=== Classification report (TEST) — Logistic Regression ===
              precision    recall  f1-score   support

         BCC      0.674     0.707     0.690       123
         ACK      0.689     0.729     0.709        85
         SEK      0.890     0.898     0.894       108
         SCC      0.353     0.400     0.375        45
         MEL      0.810     0.567     0.667        60

    accuracy                          0.708       421
   macro avg      0.683     0.660     0.667       421
weighted avg      0.718     0.708     0.709       421

=== ROC-AUC (TEST) — Logistic Regression ===
BCC: 0.871  (95% CI: 0.791–0.930)
ACK: 0.931  (95% CI: 0.895–0.960)
SEK: 0.982  (95% CI: 0.963–0.993)
SCC: 0.776  (95% CI: 0.656–0.881)
MEL: 0.874  (95% CI: 0.774–0.986)
Macro AUC: 0.887  (95% CI: 0.842–0.929)
